# Causal inference examples 

Examples from the paper
to be used in exercises in Sec 4.5.

In [1]:
# Load Python modules
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
# Simple float __repr__  
if int(np.__version__.split(".")[0]) >= 2:
    np.set_printoptions(legacy='1.25')

In [3]:
%pip install --quiet rpy2 

Note: you may need to restart the kernel to use updated packages.


In [4]:
%load_ext rpy2.ipython

## Example 1: Adjusting causes bias

Example 1 from [Why We Should Teach Causal Inference: Examples in Linear Regression With Simulated Data](https://www.tandfonline.com/doi/full/10.1080/10691898.2020.1752859)

In [5]:
%%R
# Sample Size
n <- 1000

# Set seed for reproducibility
set.seed(1896)

# Generate the observations
learning <- rnorm(n)
knowing <- 5 * learning + rnorm(n)
understanding <- 3 * knowing + rnorm(n)

In [6]:
# ALT. in Python
# np.random.seed(1896)
# n = 1000
# learns = norm(0,1).rvs(n)
# knows = 5*learns + norm(0,1).rvs(n)
# undstds = 3*knows + norm(0,1).rvs(n)
# learning1 = pd.DataFrame({"learn":learns,
#                      "know": knows,
#                      "undstd": undstds})

In [7]:
# %%R
# # Package as an R data frame
# learning1 <- data.frame(learn = learning,
#                         know = knowing,
#                         undstd = understanding)

# # Save the data frame as a CSV file
# write.csv(learning1, "../datasets/learning1.csv", row.names = FALSE)

In [8]:
# Load the dataset
learning1 = pd.read_csv("../datasets/learning1.csv")

In [9]:
import statsmodels.formula.api as smf
lm1a = smf.ols("undstd ~ 1 + learn", data=learning1).fit()
lm1a.params

Intercept    -0.022277
learn        15.120876
dtype: float64

In [10]:
lm1b = smf.ols("undstd ~ 1 + learn + know", data=learning1).fit()
lm1b.params

Intercept   -0.004948
learn        0.121515
know         2.981736
dtype: float64

In [11]:
# from ministats import plot_partreg
# plot_partreg(lm1b, pred="learn");

## Example 2:  Adjusting removes bias

Example 2 from [Why We Should Teach Causal Inference: Examples in Linear Regression With Simulated Data](https://www.tandfonline.com/doi/full/10.1080/10691898.2020.1752859)

In [12]:
%%R
set.seed(1896) # Reproducibility
n <- 1000 # Sample Size
intelligence <- rnorm(n, mean = 100, sd = 15)
learning.time <- 200 - intelligence + rnorm(n)
test.score <- 0.5 * intelligence + 0.1 * learning.time + rnorm(n)

In [13]:
# ALT. in Python
# np.random.seed(1897)
# n = 1000
# iqs = norm(100,15).rvs(n)
# ltimes = 200 - iqs + norm(0,1).rvs(n) 
# tscores = 0.5*iqs + 0.1*ltimes + norm(0,1).rvs(n)
# bdf2 = pd.DataFrame({"iq":iqs,
#                      "ltime": ltimes,
#                      "tscore": tscores})

In [14]:
# %%R
# # Package as an R data frame
# learning2 <- data.frame(iq = intelligence,
#                         ltime = learning.time,
#                         tscore = test.score)
# # Save the data frame as a CSV file
# write.csv(learning2, "../datasets/learning2.csv", row.names = FALSE)

In [15]:
# Load the dataset
learning2 = pd.read_csv("../datasets/learning2.csv")

In [16]:
lm2a = smf.ols("tscore ~ 1 + ltime", data=learning2).fit()
lm2a.params

Intercept    100.083133
ltime         -0.400917
dtype: float64

In [17]:
lm2b = smf.ols("tscore ~ 1 + ltime + iq", data=learning2).fit()
lm2b.params

Intercept    3.446496
ltime        0.081736
iq           0.483749
dtype: float64

## Example 3: Adjusting causes bias

Example 3 from [Why We Should Teach Causal Inference: Examples in Linear Regression With Simulated Data](https://www.tandfonline.com/doi/full/10.1080/10691898.2020.1752859)

In [18]:
%%R
set.seed(1896) # Reproducibility
n <- 1000 # Sample Size
network <- rnorm(n)
competence <- rnorm(n)
promotion <- ((network > 1) | (competence > 1))
luck <- rbinom(n, size = 1, prob = 0.05)
promotion <- (1 - luck) * promotion + luck * (1 - promotion)

In [19]:
# ALT. in Python
# np.random.seed(42)
# n = 1000
# ntwrks = norm(0,1).rvs(n)
# comps = norm(0,1).rvs(n)
# boths = ((ntwrks > 1) | (comps > 1))
# lucks = bernoulli(0.05).rvs(n)
# proms = (1 - lucks)*boths + lucks*(1 - boths)
# bdf3 = pd.DataFrame({"ntwrk": ntwrks,
#                      "comp": comps,
#                      "prom": proms})

In [20]:
# %%R
# # Package as an R data frame
# learning3 <- data.frame(ntwrk = network,
#                         comp = competence,
#                         prom = promotion)

# # Save the data frame as a CSV file
# write.csv(learning3, "../datasets/learning3.csv", row.names = FALSE)

Without adjusting for the collider `prom`,
there is almost no effect of the network ability on competence.

In [21]:
learning3 = pd.read_csv("../datasets/learning3.csv")

lm3a = smf.ols("comp ~ 1 + ntwrk", data=learning3).fit()
lm3a.params

Intercept   -0.005812
ntwrk        0.030412
dtype: float64

But with adjustment for `prom`,
there seems to be a negative effect.

In [22]:
lm3b = smf.ols("comp ~ 1 + ntwrk + prom", data=learning3).fit()
lm3b.params

Intercept   -0.306837
ntwrk       -0.139514
prom         0.956717
dtype: float64

The false negative effect can also appear from sampling bias,
e.g. if we restrict our analysis only to people who were promoted.

In [23]:
promoted = learning3[learning3["prom"]==1]
lm3c = smf.ols("comp ~ ntwrk", data=promoted).fit()
lm3c.params

Intercept    0.742374
ntwrk       -0.313249
dtype: float64

## Example 4: Randomized experiments

Example 4 from [Why We Should Teach Causal Inference: Examples in Linear Regression With Simulated Data](https://www.tandfonline.com/doi/full/10.1080/10691898.2020.1752859)

In [24]:
%%R
n <- 1000 # Sample Size

# Repeat code from Example 2
set.seed(1896) # Reproducibility
intelligence <- rnorm(n, mean = 100, sd = 15)

set.seed(1896) # Reproducibility
exper.group <- rbinom(n, size = 1, prob = 0.5)
learning.time.exper <- exper.group * 80 + (1 - exper.group) * 120
test.score.exper <- 0.5 * intelligence + 0.1 * learning.time.exper + rnorm(n)

In [25]:
# ALT. in Python
# np.random.seed(1896)
# n = 1000
# iqs = norm(100,15).rvs(n)
# groups = bernoulli(p=0.5).rvs(n)
# ltimes = 80*groups + 120*(1-groups)
# tscores = 0.5*iqs + 0.1*ltimes + norm(0,1).rvs(n)
# bdf4 = pd.DataFrame({"iq":iqs,
#                      "ltime": ltimes,
#                      "tscore": tscores})

In [26]:
# %%R
# # Package as an R data frame
# learning4 <- data.frame(iq = intelligence,
#                         group = exper.group,
#                         ltime = learning.time.exper,
#                         tscore = test.score.exper)

# # Save the data frame as a CSV file
# write.csv(learning4, "../datasets/learning4.csv", row.names = FALSE)

In [27]:
# Load the dataset
learning4 = pd.read_csv("../datasets/learning4.csv")

In [28]:
# non-randomized                  # random assignment
learning2.corr()["iq"]["ltime"],  learning4.corr()["iq"]["ltime"]

(-0.9978533730688123, 0.015003047523471279)

In [29]:
learning4.groupby("ltime")["iq"].mean()

ltime
80     99.387386
120    99.831793
Name: iq, dtype: float64

In [30]:
lm4a = smf.ols("tscore ~ 1 + ltime", data=learning4).fit()
lm4a.params

Intercept    49.079347
ltime         0.107285
dtype: float64

In [31]:
lm4b = smf.ols("tscore ~ 1 + ltime + iq", data=learning4).fit()
lm4b.params

Intercept   -0.028577
ltime        0.101746
iq           0.498565
dtype: float64

In [32]:
lm4c = smf.ols("tscore ~ 1 + iq", data=learning4).fit()
lm4c.params

Intercept    9.924416
iq           0.500626
dtype: float64